# Un millón de dólares contra el abandono escolar

**El problema.** Somos el equipo técnico del MINEDUC a fines de 2021. Hay que repartir US$ 1.000.000 entre los alumnos de la foto 2021 para evitar la mayor cantidad de abandonos en 2022. Invertir `m` dólares en un alumno baja su riesgo en `efecto(m) = 0,6 · (1 − e^(−m/200))`.

**Cómo leer este notebook.** Las celdas marcadas con 🟨 **DECISIÓN** son puntos donde decido yo antes de seguir. Antes de cada una, el notebook pone números sobre la mesa.

| Sección | Estado |
|---|---|
| 1. Datos y población | ✅ |
| 2. FE de la clase | ✅ |
| 3. FE nuevo (cinco grupos) | ✅ |
| 4. Prueba de fuga | ✅ |
| 5. Backtests: ¿qué suma cada grupo? | ✅ |
| 🟨 6. Decisión: qué features quedan | **pendiente** |
| 7. Modelo final 2015–2020 | — |
| 8. Escenarios de calibración | — |
| 9. Reparto del millón y entrega | — |

**Decisiones ya tomadas**
- Criterio: maximizar los abandonos evitados, sin restricciones adicionales.
- El escenario de 2022 se elige con los datos (sección 8), sin usar información externa.

## 1. Datos y población

Juntamos las fotos 2014–2020 (con etiqueta) y la foto 2021 (sin etiqueta) en una sola tabla. Es necesario: el FE de 2021 necesita la historia de cada alumno en 2020, 2019, etc.

In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DATOS = Path("../predicciones/clase6/desafio_ministerio")
SEMILLA = 102191
SIT_FIN = {"P": 0, "R": 1, "Y": 2}

d = pd.concat([pd.read_csv(DATOS / "chile_2014_2020.csv.gz"),
               pd.read_csv(DATOS / "chile_2021.csv.gz")], ignore_index=True)
d["sit_fin"] = d["sit_fin"].map(SIT_FIN)
d = d.sort_values(["id_alumno", "agno"]).reset_index(drop=True)

d.groupby("agno").agg(filas=("id_alumno", "size"), egresan=("egresa", "sum"),
                      tasa_abandono=("abandono_t1", "mean")).round(4)

,filas,egresan,tasa_abandono
agno,,,
2014,286682,18629,0.0143
2015,286747,19082,0.0143
2016,286994,19062,0.0134
2017,288367,18875,0.0139
2018,291259,19768,0.0146
2019,293716,19753,0.0183
2020,293882,20575,0.0114
2021,298173,20632,NaN


- 2021 no tiene tasa porque su etiqueta mira 2022, que es justamente lo que se evalúa.
- Los egresados no pueden abandonar: se sacan para modelar y en 2021 reciben monto 0. El FE se calcula **antes** de sacarlos, con la historia completa.

## 2. El FE de la clase

Son las 11 variables de `abandono_chile/python/fe.py`, reescritas acá para poder modificarlas. Todas miran hacia atrás (el mismo alumno en años anteriores) o al mismo año t.

Un cambio respecto del original: `repitencias_previas` se calcula con una suma acumulada en lugar de un `lambda` por alumno. Da lo mismo y es mucho más rápido.

In [2]:
CRUDAS = ["cod_reg_rbd", "cod_com_rbd", "cod_depe2", "rural_rbd", "cod_ense2", "cod_grado", "cod_jor",
          "cod_rama", "cod_espe", "gen_alu", "edad", "prom_gral", "asistencia", "sit_fin", "n_reg",
          "cambio_esc_anio", "esc_n", "esc_prom", "esc_asist", "esc_rep", "esc_ab_lag"]

FE_CLASE = ["prom_gral_lag1", "asistencia_lag1", "sit_fin_lag1", "prom_gral_delta1", "asistencia_delta1",
            "repitencias_previas", "anios_previos", "cambio_esc_entre", "sobreedad", "prom_vs_esc",
            "asist_vs_esc"]


def fe_clase(d):
    g = d.groupby("id_alumno", sort=False)
    seguido = g["agno"].shift(1) == d["agno"] - 1
    for v in ["prom_gral", "asistencia", "sit_fin"]:
        d[f"{v}_lag1"] = g[v].shift(1).where(seguido)
    for v in ["prom_gral", "asistencia"]:
        d[f"{v}_delta1"] = d[v] - d[f"{v}_lag1"]
    rep = (d["sit_fin"] == 1).astype(int)
    d["repitencias_previas"] = rep.groupby(d["id_alumno"]).cumsum() - rep
    d["anios_previos"] = g.cumcount()
    d["cambio_esc_entre"] = (g["id_escuela"].shift(1) != d["id_escuela"]).astype(float).where(seguido)
    d["sobreedad"] = d["edad"] - d.groupby(["agno", "cod_ense2", "cod_grado"])["edad"].transform("median")
    d["prom_vs_esc"] = d["prom_gral"] - d["esc_prom"]
    d["asist_vs_esc"] = d["asistencia"] - d["esc_asist"]
    return d

## 3. FE nuevo: cinco grupos, cada uno con su porqué

Cada grupo sale de algo que aprendimos en la clase. Se pueden prender y apagar por separado, así en la sección 5 se mide qué suma cada uno.

| Grupo | Qué hace | De dónde sale la idea |
|---|---|---|
| **A · Retirados sin ceros falsos** | Para `sit_fin = Y`, promedio y asistencia pasan de 0 a NaN *antes* de calcular lags y deltas | El 0 del retirado es "sin dato": ensucia `asistencia_delta1` (el −77) y el lag del año siguiente |
| **B · Huecos y abandonos previos** | `abandonos_previos`, `huecos_previos`, `anios_desde_previo` | En 2019, nada del FE le decía al modelo "este chico ya dejó la escuela una vez" |
| **C · Relativas al año** | Percentil de asistencia y promedio dentro del año, y dentro del año y grado | En 2020 la asistencia cambió de nivel (remoto). El percentil mira la posición, no el valor |
| **D · Escuela en movimiento** | Cambio de asistencia, promedio y tamaño de la escuela respecto del año anterior | Separar "mi escuela cerró o se achicó" de "yo falté" (estallido, pandemia) |
| **E · Trayectoria larga** | Asistencia y promedio de t−2 | Un solo año de historia puede ser ruido; dos años muestran tendencia |

Y una que se **saca**: `anios_previos`, que deriva con el calendario (train 1–2, test 4). Se evalúa con y sin ella.

### Sobre la fuga en B
`abandonos_previos` usa `abandono_t1` de **filas anteriores** del mismo alumno. El de la fila t−1 se conoce en t (es si estaba en t), así que en el año t ya está disponible. El de la propia fila t no se usa nunca: por eso se resta en el código. En las filas de 2021 los abandonos de 2020 hacia atrás ya se conocen.

In [3]:
def fe_nuevo(d, A=False, B=False, C=False, D=False, E=False):
    d = d.copy()
    if A:
        d.loc[d["sit_fin"] == 2, ["prom_gral", "asistencia"]] = np.nan
    d = fe_clase(d)
    g = d.groupby("id_alumno", sort=False)
    previo = g["agno"].shift(1)
    nuevas = []

    if B:
        ab = d["abandono_t1"].fillna(0)
        d["abandonos_previos"] = ab.groupby(d["id_alumno"]).cumsum() - ab
        hueco = ((d["agno"] - previo) > 1).astype(int)
        d["huecos_previos"] = hueco.groupby(d["id_alumno"]).cumsum()
        d["anios_desde_previo"] = d["agno"] - previo
        nuevas += ["abandonos_previos", "huecos_previos", "anios_desde_previo"]

    if C:
        for v in ["asistencia", "prom_gral"]:
            d[f"{v}_pct_anio"] = d.groupby("agno")[v].rank(pct=True)
            d[f"{v}_pct_grado"] = d.groupby(["agno", "cod_ense2", "cod_grado"])[v].rank(pct=True)
            nuevas += [f"{v}_pct_anio", f"{v}_pct_grado"]

    if D:
        e = (d.groupby(["id_escuela", "agno"], as_index=False)[["esc_asist", "esc_prom", "esc_n"]].first()
              .sort_values(["id_escuela", "agno"]))
        ge = e.groupby("id_escuela")
        sig = ge["agno"].shift(1) == e["agno"] - 1
        e["esc_asist_delta1"] = (e["esc_asist"] - ge["esc_asist"].shift(1)).where(sig)
        e["esc_prom_delta1"] = (e["esc_prom"] - ge["esc_prom"].shift(1)).where(sig)
        e["esc_n_var1"] = (e["esc_n"] / ge["esc_n"].shift(1) - 1).where(sig)
        cols = ["esc_asist_delta1", "esc_prom_delta1", "esc_n_var1"]
        d = d.merge(e[["id_escuela", "agno"] + cols], on=["id_escuela", "agno"], how="left").set_index(d.index)
        nuevas += cols

    if E:
        seguido2 = (g["agno"].shift(2) == d["agno"] - 2) & (previo == d["agno"] - 1)
        for v in ["asistencia", "prom_gral"]:
            d[f"{v}_lag2"] = g[v].shift(2).where(seguido2)
            nuevas.append(f"{v}_lag2")

    return d, nuevas

In [4]:
d_raw, nuevas_raw = fe_nuevo(d, A=False, B=True, C=True, D=True, E=True)
d_A, nuevas_A = fe_nuevo(d, A=True, B=True, C=True, D=True, E=True)

GRUPOS = {"B": ["abandonos_previos", "huecos_previos", "anios_desde_previo"],
          "C": ["asistencia_pct_anio", "asistencia_pct_grado", "prom_gral_pct_anio", "prom_gral_pct_grado"],
          "D": ["esc_asist_delta1", "esc_prom_delta1", "esc_n_var1"],
          "E": ["asistencia_lag2", "prom_gral_lag2"]}
assert sorted(sum(GRUPOS.values(), [])) == sorted(nuevas_raw)
len(d_raw), len(nuevas_raw)

(2325820, 12)

### Un vistazo: el alumno del hueco

Es el mismo tipo de caso que vimos en el bloque 2. Mirá qué ve ahora el modelo en la fila posterior al hueco: `abandonos_previos` y `huecos_previos` dicen lo que antes había que adivinar a partir de los NaN.

In [5]:
ejemplo = d_A[(d_A["huecos_previos"] > 0) & (d_A["abandonos_previos"] > 0) & (d_A["agno"] == 2019)]["id_alumno"].iloc[0]
cols_ver = ["agno", "id_escuela", "cod_grado", "edad", "prom_gral", "asistencia", "sit_fin", "abandono_t1",
            "asistencia_lag1", "asistencia_delta1", "abandonos_previos", "huecos_previos",
            "asistencia_pct_anio", "esc_asist_delta1", "asistencia_lag2"]
d_A.loc[d_A["id_alumno"] == ejemplo, cols_ver].round(2)

,agno,id_escuela,cod_grado,edad,prom_gral,asistencia,sit_fin,abandono_t1,asistencia_lag1,asistencia_delta1,abandonos_previos,huecos_previos,asistencia_pct_anio,esc_asist_delta1,asistencia_lag2
738,2014,E05841,3,8,6.1,86.0,0.0,0.0,NaN,NaN,0.0,0,0.17,NaN,NaN
739,2015,E04380,4,9,6.4,81.0,0.0,1.0,86.0,-5.0,0.0,0,0.08,-0.30,NaN
740,2018,E07750,7,12,5.2,94.0,0.0,0.0,NaN,NaN,1.0,1,0.50,-1.06,NaN
741,2019,E05604,8,13,5.9,98.0,0.0,0.0,94.0,4.0,1.0,1,0.83,-1.61,NaN
742,2020,E03911,1,14,5.1,100.0,0.0,0.0,98.0,2.0,1.0,1,0.77,24.65,94.0
743,2021,E03911,2,15,4.4,9.0,1.0,NaN,100.0,-91.0,1.0,1,0.01,-92.74,98.0


## 4. Prueba de fuga

Es la prueba más directa: si calculo el FE **cortando los datos en 2018** y lo comparo con el FE calculado con **todos** los años, las filas de 2018 tienen que dar idénticas. Si alguna variable cambia, es que mira el futuro.

Ojo con lo que **no** prueba: que `abandonos_previos` use etiquetas pasadas está bien porque ya se conocen en t. Esta prueba detecta futuro, no uso de etiquetas.

In [6]:
cols_todas = FE_CLASE + nuevas_A
cortado, _ = fe_nuevo(d[d["agno"] <= 2018], A=True, B=True, C=True, D=True, E=True)
fa = cortado[cortado["agno"] == 2018].set_index("id_alumno")[cols_todas].sort_index()
fb = d_A[d_A["agno"] == 2018].set_index("id_alumno")[cols_todas].sort_index()
pd.testing.assert_frame_equal(fa, fb)
"sin fuga: las filas de 2018 no cambian al agregar 2019–2021"

'sin fuga: las filas de 2018 no cambian al agregar 2019–2021'

## 5. Backtests: ¿qué suma cada grupo?

Dos backtests, con la lógica de la clase:

| Backtest | Entrena | Evalúa | Para qué |
|---|---|---|---|
| **normal** | 2015–2017 | 2018 | El año "típico" de la clase |
| **pandemia** | 2015–2019 | 2020 | El año con etiqueta más parecido al que hay que predecir (2021 → 2022) |

**Hiperparámetros fijos:** los que eligió la búsqueda bayesiana en mi corrida de la clase. La cantidad de árboles se escala con las filas, como en `modelo_final`. No se vuelve a tunear: la búsqueda sumó 0,005 de AUC y el FE 0,046, así que el esfuerzo va primero al FE.

**Dos métricas:**
- **AUC**: qué tan bien ordena a todos.
- **Captura top 5 %**: de todos los que abandonan, qué fracción queda dentro del 5 % de alumnos más riesgosos según el modelo. Es la zona donde va a ir la plata.

In [7]:
PARAMS = {"objective": "binary", "verbose": -1, "seed": SEMILLA, "learning_rate": 0.03295,
          "num_leaves": 66, "min_data_in_leaf": 811, "feature_fraction": 0.6293, "lambda_l2": 4.786,
          "num_threads": 0}
ARBOLES_POR_FILA = 226 / 535_597
BACKTESTS = {"normal 2018": (2015, 2017, 2018), "pandemia 2020": (2015, 2019, 2020)}


def poblacion(dd):
    return dd[dd["egresa"] == 0]


def entrenar(dd, cols, desde, hasta):
    p = poblacion(dd)
    tr = p[p["agno"].between(desde, hasta)]
    n = max(1, round(ARBOLES_POR_FILA * len(tr)))
    return lgb.train(PARAMS, lgb.Dataset(tr[cols], tr["abandono_t1"]), n)


def captura_top(y, s, frac=0.05):
    top = np.argsort(-s)[: int(len(s) * frac)]
    return y[top].sum() / y.sum()


def evaluar(dd, cols):
    fila = {}
    for nombre, (desde, hasta, test) in BACKTESTS.items():
        m = entrenar(dd, cols, desde, hasta)
        te = poblacion(dd)
        te = te[te["agno"] == test]
        y, s = te["abandono_t1"].to_numpy(), m.predict(te[cols])
        fila[f"AUC {nombre}"] = roc_auc_score(y, s)
        fila[f"top5 {nombre}"] = captura_top(y, s)
    return fila

In [8]:
BASE = CRUDAS + FE_CLASE
SIN_AP = [c for c in BASE if c != "anios_previos"]
TODO = BASE + sum(GRUPOS.values(), [])

configs = {
    "clase (base)": (d_raw, BASE),
    "base − anios_previos": (d_raw, SIN_AP),
    "base + A retirados NaN": (d_A, BASE),
    "base + B huecos": (d_raw, BASE + GRUPOS["B"]),
    "base + C relativas": (d_raw, BASE + GRUPOS["C"]),
    "base + D escuela": (d_raw, BASE + GRUPOS["D"]),
    "base + E lag2": (d_raw, BASE + GRUPOS["E"]),
    "todo (A–E)": (d_A, TODO),
    "todo − anios_previos": (d_A, [c for c in TODO if c != "anios_previos"]),
}

filas = []
for nombre, (dd, cols) in configs.items():
    filas.append({"config": nombre, "n_vars": len(cols), **evaluar(dd, cols)})
    print(f"{nombre:28s} listo", flush=True)

ablacion = pd.DataFrame(filas).set_index("config")
ablacion

clase (base)                 listo


base − anios_previos         listo


base + A retirados NaN       listo


base + B huecos              listo


base + C relativas           listo


base + D escuela             listo


base + E lag2                listo


todo (A–E)                   listo


todo − anios_previos         listo


,n_vars,AUC normal 2018,top5 normal 2018,AUC pandemia 2020,top5 pandemia 2020
config,,,,,
clase (base),32,0.923752,0.719022,0.862970,0.531820
base − anios_previos,31,0.918028,0.709852,0.839783,0.492082
base + A retirados NaN,32,0.923466,0.717141,0.862727,0.534210
base + B huecos,35,0.923716,0.720903,0.862685,0.530624
base + C relativas,36,0.924146,0.720198,0.871697,0.540185
base + D escuela,35,0.923822,0.720903,0.861395,0.524649
base + E lag2,34,0.924268,0.719492,0.864114,0.529131
todo (A–E),44,0.924689,0.721608,0.869970,0.541679
todo − anios_previos,43,0.923414,0.721373,0.860242,0.526740


In [9]:
base = ablacion.loc["clase (base)"]
dif = (ablacion.drop(columns="n_vars") - base.drop("n_vars")).round(4)
dif.columns = [f"Δ {c}" for c in dif.columns]
dif

,Δ AUC normal 2018,Δ top5 normal 2018,Δ AUC pandemia 2020,Δ top5 pandemia 2020
config,,,,
clase (base),0.0000,0.0000,0.0000,0.0000
base − anios_previos,-0.0057,-0.0092,-0.0232,-0.0397
base + A retirados NaN,-0.0003,-0.0019,-0.0002,0.0024
base + B huecos,-0.0000,0.0019,-0.0003,-0.0012
base + C relativas,0.0004,0.0012,0.0087,0.0084
base + D escuela,0.0001,0.0019,-0.0016,-0.0072
base + E lag2,0.0005,0.0005,0.0011,-0.0027
todo (A–E),0.0009,0.0026,0.0070,0.0099
todo − anios_previos,-0.0003,0.0024,-0.0027,-0.0051


**Cómo leer las diferencias.** Un Δ AUC de +0,002 puede ser ruido. Como referencia, en la clase la búsqueda de hiperparámetros sumó +0,005 y el FE completo +0,046. Para tomar una decisión, conviene mirar si el signo se repite en los **dos** backtests y en las **dos** métricas.

### ¿Qué usa el modelo con todo?

Importancia por ganancia del modelo "todo (A–E)" en el backtest de pandemia. Las variables nuevas están marcadas. Recordatorio: **importancia no es efecto**, y variables que dicen lo mismo se reparten la importancia entre ellas.

In [10]:
m_todo = entrenar(d_A, TODO, 2015, 2019)
imp = pd.DataFrame({"variable": TODO, "ganancia": m_todo.feature_importance("gain")})
imp["ganancia"] = imp["ganancia"] / imp["ganancia"].sum()
imp["grupo"] = imp["variable"].map({v: g for g, vs in GRUPOS.items() for v in vs}).fillna(
    imp["variable"].map(lambda v: "FE clase" if v in FE_CLASE else "cruda"))
imp.sort_values("ganancia", ascending=False).head(20).round(4).reset_index(drop=True)

,variable,ganancia,grupo
0,prom_gral,0.1805,cruda
1,prom_gral_pct_anio,0.1255,C
2,asistencia_delta1,0.0784,FE clase
3,sit_fin,0.0697,cruda
4,asistencia_pct_anio,0.0621,C
5,asistencia,0.0599,cruda
6,esc_ab_lag,0.0583,cruda
7,edad,0.0452,cruda
8,cod_grado,0.0373,cruda
9,asist_vs_esc,0.0312,FE clase


## 🟨 6. DECISIÓN: ¿qué features quedan?

Con la tabla de ablación y la de importancia, hay que decidir:

1. **¿Qué grupos entran (A, B, C, D, E)?** ¿Cuál fue el criterio: el backtest de pandemia, el normal o los dos?
2. **¿`anios_previos` queda o sale?** Acordate de la deriva: en 2021 va a valer hasta 7.
3. **¿Qué métrica pesa más, AUC o captura top 5 %?** La plata va a ir arriba de todo.
4. **¿Algún grupo suma en un backtest y resta en el otro?** ¿Cómo lo interpretás con lo que sabemos de 2020?

Lo que sigue depende de esta decisión:
- **7. Modelo final:** entrena 2015–2020 con las features elegidas y predice la foto 2021.
- **8. Escenarios:** calibración "como 2020" (isotónica con el backtest de pandemia), "como 2019" y "normal". Se mide cuánto cambia el reparto entre escenarios.
- **9. Reparto:** asignación óptima con rendimientos decrecientes (el próximo dólar va a quien más rinde), entrega en CSV y validaciones de la consigna.